In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

In [2]:
options = Options()
options.add_argument("--headless")  
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [3]:
base_url = "https://www.transfermarkt.com"
start_url = f"{base_url}/spieler-statistik/wertvollstemannschaften/marktwertetop"
driver.get(start_url)
time.sleep(5) 

In [4]:
soup = BeautifulSoup(driver.page_source, "html.parser")
teams = []

In [5]:
rows = soup.select("table.items tbody tr")
for row in rows:
    team_cell = row.select_one("td.hauptlink a")
    if team_cell:
        name = team_cell.text.strip()
        relative_link = team_cell.get("href")
        full_link = base_url + relative_link
        teams.append((name, full_link))

print(f"✅ Found {len(teams)} teams.")

✅ Found 25 teams.


In [6]:
def scrape_team_details(url):
    try:
        driver.get(url)
        time.sleep(3)  

        soup = BeautifulSoup(driver.page_source, "html.parser")
        data = {}

       
        team_name_tag = soup.select_one("header h1.data-header__headline-wrapper")
        data["Team"] = team_name_tag.get_text(strip=True) if team_name_tag else None

        
        info_box = soup.select_one("header div.data-header__info-box")
        if info_box:
            for li in info_box.find_all("li"):
                text = " ".join(li.stripped_strings)
                if "Coach" in text:
                    data["Coach"] = text.split(":")[-1].strip()
                elif "Stadium" in text:
                    data["Stadium"] = text.split(":")[-1].strip()
                elif "Squad size" in text:
                    data["Squad Size"] = text.split(":")[-1].strip()
                elif "Average age" in text:
                    data["Average Age"] = text.split(":")[-1].strip()
                elif "Foreigners" in text:
                    data["Foreigners"] = text.split(":")[-1].strip()
                elif "Total market value" in text:
                    data["Total Market Value"] = text.split(":")[-1].strip()

        data["URL"] = url
        return data

    except Exception as e:
        print(f"❌ Error scraping {url}: {e}")
        return None

In [7]:
records = []
for name, url in teams:
    print(f"Scraping {name} ...")
    team_data = scrape_team_details(url)
    if team_data:
        records.append(team_data)
        print(f"{team_data.get('Team', name)} scraped successfully")
    else:
        print(f"Failed to scrape {name}")
    time.sleep(2)  

driver.quit()

Scraping Real Madrid ...
Real Madrid scraped successfully
Scraping Arsenal FC ...
Arsenal FC scraped successfully
Scraping Manchester City ...
Manchester City scraped successfully
Scraping Liverpool FC ...
Liverpool FC scraped successfully
Scraping Paris Saint-Germain ...
Paris Saint-Germain scraped successfully
Scraping Chelsea FC ...
Chelsea FC scraped successfully
Scraping FC Barcelona ...
FC Barcelona scraped successfully
Scraping Bayern Munich ...
Bayern Munich scraped successfully
Scraping Tottenham Hotspur ...
Tottenham Hotspur scraped successfully
Scraping Newcastle United ...
Newcastle United scraped successfully
Scraping Manchester United ...
Manchester United scraped successfully
Scraping Inter Milan ...
Inter Milan scraped successfully
Scraping Nottingham Forest ...
Nottingham Forest scraped successfully
Scraping Atlético de Madrid ...
Atlético de Madrid scraped successfully
Scraping Juventus FC ...
Juventus FC scraped successfully
Scraping Aston Villa ...
Aston Villa scrap

In [8]:
df = pd.DataFrame(records)
print(df.head())

                  Team Squad Size Average Age Foreigners  \
0          Real Madrid         25        25.9  18 72.0 %   
1           Arsenal FC         25        25.7  17 68.0 %   
2      Manchester City         26        26.4  19 73.1 %   
3         Liverpool FC         26        25.8  20 76.9 %   
4  Paris Saint-Germain         24        23.6  14 58.3 %   

                          Stadium  \
0  Santiago Bernabéu 83.186 Seats   
1   Emirates Stadium 60.704 Seats   
2     Etihad Stadium 55.097 Seats   
3            Anfield 61.276 Seats   
4   Parc des Princes 48.583 Seats   

                                                 URL  
0  https://www.transfermarkt.com/real-madrid/star...  
1  https://www.transfermarkt.com/fc-arsenal/start...  
2  https://www.transfermarkt.com/manchester-city/...  
3  https://www.transfermarkt.com/fc-liverpool/sta...  
4  https://www.transfermarkt.com/fc-paris-saint-g...  


In [9]:
foreigners_split = df["Foreigners"].str.split(" ", n=1, expand=True)
df["Foreigners Count"] = pd.to_numeric(foreigners_split[0], errors="coerce")  
df["Foreigners %"] = foreigners_split[1].str.replace("%", "", regex=False)
df["Foreigners %"] = pd.to_numeric(df["Foreigners %"], errors="coerce")


stadium_split = df["Stadium"].str.rsplit(" ", n=2, expand=True)
df["Stadium Name"] = stadium_split[0]
df["Stadium Capacity"] = stadium_split[1].str.replace(".", "", regex=False)
df["Stadium Capacity"] = pd.to_numeric(df["Stadium Capacity"], errors="coerce")


df = df.drop(columns=["Foreigners", "Stadium"])
cols = [c for c in df.columns if c != "URL"] + ["URL"]
df = df[cols]

print(df)


                      Team Squad Size Average Age  Foreigners Count  \
0              Real Madrid         25        25.9                18   
1               Arsenal FC         25        25.7                17   
2          Manchester City         26        26.4                19   
3             Liverpool FC         26        25.8                20   
4      Paris Saint-Germain         24        23.6                14   
5               Chelsea FC         31        23.5                21   
6             FC Barcelona         23        25.5                10   
7            Bayern Munich         25        26.7                12   
8        Tottenham Hotspur         29        25.3                23   
9         Newcastle United         29        27.4                13   
10       Manchester United         26        25.6                19   
11             Inter Milan         25        28.6                17   
12       Nottingham Forest         28        26.3                21   
13    